# AMG-Colorization Demo

Interactive demonstration of the Multi-GAN colorization pipeline:
- Image preprocessing (edge detection, segmentation, clustering)
- Training on CIFAR-10
- Single image colorization
- Video colorization with temporal harmony

## 1. Setup & Imports

In [ ]:
import os
import sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
import torch
import cv2
from PIL import Image
from tqdm import tqdm

# Add project root to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from utils.device import get_device
from utils.visualization import plot_colorization, tensor_to_numpy
from utils.metrics import compute_psnr, compute_ssim
from data.dataset_loader import create_dataloader
from preprocessing.edge_detection import get_edge_map
from preprocessing.segmentation import segment_regions, compute_smv
from preprocessing.clustering import cluster_regions, compute_dynamic_k
from models.generator import UNetGenerator
from models.discriminator import PatchGANDiscriminator
from models.gan import GAN
from models.multi_gan import MultiGAN
from training.trainer import MultiGANTrainer
from inference.image_infer import ImageColorizer, colorize_image
from inference.video_infer import VideoColorizer

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

# Set up device
device = get_device('auto')
print(f'Using device: {device}')

## 2. Load Configuration

In [ ]:
# Load config
config_path = '../configs/default.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Print key settings
for section in ['data', 'clustering', 'training', 'inference']:
    print(f'\n[{section}]')
    for k, v in config.get(section, {}).items():
        print(f'  {k}: {v}')

## 3. Preprocessing Demonstration

Show the edge detection, segmentation, and clustering pipeline on a sample image.

In [ ]:
# Create a synthetic test image
def create_test_image(size=256):
    img = np.zeros((size, size), dtype=np.uint8)
    # Gradient background
    for i in range(size):
        img[i, :] = int(255 * i / size)
    # Add some shapes
    cv2.rectangle(img, (50, 50), (100, 100), color=200, thickness=-1)
    cv2.circle(img, (180, 150), 40, color=80, thickness=-1)
    cv2.ellipse(img, (120, 200), (30, 20), 0, 0, 360, color=160, thickness=-1)
    return img

test_img = create_test_image()
plt.imshow(test_img, cmap='gray')
plt.title('Test Grayscale Image')
plt.axis('off')
plt.show()

In [ ]:
# Edge detection
edges = get_edge_map(test_img, method='canny')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(test_img, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(edges, cmap='gray')
axes[1].set_title('Canny Edges')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Region segmentation
regions = segment_regions(edges, min_region_size=64)
gray_float = test_img.astype(np.float32) / 255.0
compute_smv(gray_float, regions)

print(f'Found {len(regions)} regions')
for i, r in enumerate(regions[:5]):
    print(f'  Region {i}: area={r.area}, SMV={r.smv:.3f}, bbox={r.bbox}')

# Visualize regions with random colors
region_vis = np.zeros((*test_img.shape, 3), dtype=np.uint8)
for r in regions:
    color = np.random.randint(0, 255, 3)
    region_vis[r.mask] = color

plt.imshow(region_vis)
plt.title(f'Segmented Regions (n={len(regions)})')
plt.axis('off')
plt.show()

In [ ]:
# Clustering
k = compute_dynamic_k(test_img, gamma=0.05, k_min=10, k_max=64)
print(f'Dynamic K: {k}')

smv_values = np.array([r.smv for r in regions], dtype=np.float32)
labels, kmeans = cluster_regions(smv_values, k=k)
for i, r in enumerate(regions):
    r.cluster_id = int(labels[i])

# Show per-cluster colors
cluster_vis = np.zeros((*test_img.shape, 3), dtype=np.uint8)
cluster_colors = {}
for cid in range(k):
    cluster_colors[cid] = np.random.randint(0, 255, 3)
for r in regions:
    cluster_vis[r.mask] = cluster_colors[r.cluster_id]

plt.imshow(cluster_vis)
plt.title(f'Clustered Regions (K={k})')
plt.axis('off')
plt.show()

## 4. Model Architecture Inspection

In [ ]:
# Inspect generator
gen = UNetGenerator(in_channels=1, out_channels=3, base_features=64)
x = torch.randn(1, 1, 256, 256)
out = gen(x)
print(f'Generator output shape: {out.shape}')
print(f'Generator parameters: {sum(p.numel() for p in gen.parameters()):,}')

# Inspect discriminator
disc = PatchGANDiscriminator(in_channels=4, base_features=64)
y = torch.randn(1, 3, 256, 256)
cond = torch.randn(1, 1, 256, 256)
d_out = disc(y, cond)
print(f'Discriminator output shape: {d_out.shape}')
print(f'Discriminator parameters: {sum(p.numel() for p in disc.parameters()):,}')

## 5. Training on CIFAR-10

Train the Multi-GAN ensemble for a few epochs (adjust epochs for full training).

In [ ]:
# Create dataloaders
train_loader, val_loader = create_dataloader(
    dataset_type='cifar10',
    train_path='../data',
    val_path='../data',
    image_size=(256, 256),
    batch_size=8,
    num_workers=2,
    download=True,
)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Quick training run (5 epochs for demo)
clust_cfg = config.get('clustering', {})
num_clusters = clust_cfg.get('k_max', 64)
print(f'Creating Multi-GAN with {num_clusters} clusters...')

multi_gan = MultiGAN(
    num_clusters=num_clusters,
    device=device,
    gen_kwargs=config.get('generator', {}),
    disc_kwargs=config.get('discriminator', {}),
    lr_g=config.get('training', {}).get('lr_g', 2e-4),
    lr_d=config.get('training', {}).get('lr_d', 2e-4),
)

trainer = MultiGANTrainer(multi_gan, config, device)

# Quick demo: train for 2 epochs
config_demo = dict(config)
config_demo.setdefault('training', {})['epochs'] = 2
trainer.epochs = 2
trainer.train(train_loader, val_loader)

## 6. Image Colorization Inference

In [ ]:
# Test on a validation sample
gray_sample, color_gt = next(iter(val_loader))
gray_sample = gray_sample[0:1]  # take first image
color_gt = color_gt[0:1]

# Colorize using the Multi-GAN
with torch.no_grad():
    gray_norm = gray_sample * 2.0 - 1.0
    pred = multi_gan.generate_for_cluster(0, gray_norm)  # use first cluster GAN
    pred = (pred + 1.0) / 2.0

# Display
plot_colorization(gray_sample[0], pred[0], color_gt[0])

# Metrics
psnr_val = compute_psnr(pred, color_gt)
ssim_val = compute_ssim(pred, color_gt)
print(f'PSNR: {psnr_val:.2f} dB')
print(f'SSIM: {ssim_val:.4f}')

## 7. Video Colorization with Harmony

Demonstrate temporal color consistency across consecutive frames.

In [ ]:
# Simulate a 3-frame sequence from the same image with slight perturbation
def create_frame_sequence(base_img, n_frames=5, noise_level=0.02):
    frames = []
    for i in range(n_frames):
        noisy = base_img.astype(np.float32) + np.random.randn(*base_img.shape) * noise_level * 255
        noisy = np.clip(noisy, 0, 255).astype(np.uint8)
        frames.append(noisy)
    return frames

# Use a validation image as base frame
base_frame = (gray_sample[0, 0].numpy() * 255).astype(np.uint8)
frames = create_frame_sequence(base_frame, n_frames=3)

# Colorize with harmony
video_colorizer = VideoColorizer(multi_gan, config, device)
colorized_frames = []
for i, frame in enumerate(frames):
    result = video_colorizer.colorize_frame(frame, frame_idx=i)
    colorized_frames.append(result)

# Display results
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i in range(3):
    axes[0, i].imshow(frames[i], cmap='gray')
    axes[0, i].set_title(f'Input Frame {i+1}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(colorized_frames[i])
    axes[1, i].set_title(f'Colorized Frame {i+1}')
    axes[1, i].axis('off')
    
    if i > 0:
        diff = np.abs(colorized_frames[i].astype(float) - colorized_frames[i-1].astype(float))
        axes[2, i].imshow(diff / diff.max() if diff.max() > 0 else diff)
        axes[2, i].set_title(f'Delta (frame {i} - {i+1})' if i == 1 else f'Delta (frame {i+1} - {i})')
        axes[2, i].axis('off')

axes[2, 0].axis('off')
plt.tight_layout()
plt.show()

print('Temporal consistency check:')
for i in range(1, 3):
    mse = np.mean((colorized_frames[i].astype(float) - colorized_frames[i-1].astype(float)) ** 2)
    print(f'  Frame {i} vs Frame {i+1}: MSE = {mse:.4f}')

## 8. Metrics Evaluation

In [ ]:
# Compute metrics across validation set
psnr_values = []
ssim_values = []

for gray_batch, color_batch in tqdm(val_loader, desc='Evaluating'):
    for i in range(min(gray_batch.size(0), 4)):  # limit for speed
        gray = gray_batch[i:i+1]
        color = color_batch[i:i+1]
        with torch.no_grad():
            pred = multi_gan.generate_for_cluster(0, gray * 2.0 - 1.0)
            pred = (pred + 1.0) / 2.0
        psnr_values.append(compute_psnr(pred, color))
        ssim_values.append(compute_ssim(pred, color))

print(f'Mean PSNR: {np.mean(psnr_values):.2f} dB')
print(f'Mean SSIM: {np.mean(ssim_values):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(psnr_values, bins=20, edgecolor='black')
axes[0].set_xlabel('PSNR (dB)')
axes[0].set_title('PSNR Distribution')
axes[1].hist(ssim_values, bins=20, edgecolor='black')
axes[1].set_xlabel('SSIM')
axes[1].set_title('SSIM Distribution')
plt.tight_layout()
plt.show()